# knot — 01: deploy

Build a spec → see the SQL knot generates → execute it → look at the
tables. Subsequent notebooks (02_ingest, 03_query, ...) cover the
runtime concerns.

In [ ]:
import json
import uuid

import psycopg

from knot import Spec, types

In [ ]:
spec = Spec(identifier_slot_name="canonical_id")

person = spec.add_class("Person")
person.slot("name", types.TEXT, required=True)
person.slot("birth_country", types.TEXT)

movie = spec.add_class("Movie")
movie.slot("title", types.TEXT, required=True)
movie.slot("year", types.INTEGER)
movie.slot("director", person)  # FK — pass the class

spec

In [ ]:
# Host plumbing — psycopg connection + a fresh per-run schema so
# re-running the notebook never collides with prior runs.
pg = psycopg.connect(
    host="localhost",
    port=5433,
    user="knot",
    password="knot",
    dbname="knot",
    autocommit=True,
)
schema = f"knot_play_{uuid.uuid4().hex[:8]}"
pg.execute(f"CREATE SCHEMA {schema}")
schema